# Import & connexion

In [1]:
using MimiqCircuits

In [ ]:
conn = connect("user.name@epita.fr", "password")

---
# Functions

In [3]:
function add_surface_code_d3(c, data_qubits, ancillas, c_bits)
    for _ in 1:3
        push!(c, Reset(), ancillas)
        push!(c, GateH(), ancillas[1:8])

        # X stabilizers
        push!(c, GateCX(), ancillas[1], data_qubits[1])
        push!(c, GateCX(), ancillas[1], data_qubits[2])
        push!(c, GateCX(), ancillas[1], data_qubits[4])
        push!(c, GateCX(), ancillas[1], data_qubits[5])

        push!(c, GateCX(), ancillas[2], data_qubits[5])
        push!(c, GateCX(), ancillas[2], data_qubits[6])
        push!(c, GateCX(), ancillas[2], data_qubits[8])
        push!(c, GateCX(), ancillas[2], data_qubits[9])

        push!(c, GateCX(), ancillas[3], data_qubits[2])
        push!(c, GateCX(), ancillas[3], data_qubits[3])

        push!(c, GateCX(), ancillas[4], data_qubits[7])
        push!(c, GateCX(), ancillas[4], data_qubits[8])

        # Z stabilizers
        push!(c, GateCZ(), ancillas[5], data_qubits[2])
        push!(c, GateCZ(), ancillas[5], data_qubits[3])
        push!(c, GateCZ(), ancillas[5], data_qubits[5])
        push!(c, GateCZ(), ancillas[5], data_qubits[6])

        push!(c, GateCZ(), ancillas[6], data_qubits[4])
        push!(c, GateCZ(), ancillas[6], data_qubits[5])
        push!(c, GateCZ(), ancillas[6], data_qubits[7])
        push!(c, GateCZ(), ancillas[6], data_qubits[8])

        push!(c, GateCZ(), ancillas[7], data_qubits[1])
        push!(c, GateCZ(), ancillas[7], data_qubits[4])

        push!(c, GateCZ(), ancillas[8], data_qubits[6])
        push!(c, GateCZ(), ancillas[8], data_qubits[9])

        push!(c, GateH(), ancillas[1:8])
        push!(c, Measure(), ancillas[1:8], c_bits[1:8])

        # Z corrections when X detected
        push!(c, IfStatement(GateZ(), BitString("10000000")), data_qubits[1], c_bits[1:8]...)
        push!(c, IfStatement(GateZ(), BitString("01000000")), data_qubits[9], c_bits[1:8]...)
        push!(c, IfStatement(GateZ(), BitString("11000000")), data_qubits[5], c_bits[1:8]...)

        push!(c, IfStatement(GateZ(), BitString("00100000")), data_qubits[3], c_bits[1:8]...)
        push!(c, IfStatement(GateZ(), BitString("10100000")), data_qubits[2], c_bits[1:8]...)
        push!(c, IfStatement(GateZ(), BitString("01100000")), [data_qubits[3], data_qubits[9]], c_bits[1:8]...)
        push!(c, IfStatement(GateZ(), BitString("11100000")), [data_qubits[3], data_qubits[5]], c_bits[1:8]...)
    
        push!(c, IfStatement(GateZ(), BitString("00010000")), data_qubits[7], c_bits[1:8]...)
        push!(c, IfStatement(GateZ(), BitString("10010000")), [data_qubits[1], data_qubits[7]], c_bits[1:8]...)
        push!(c, IfStatement(GateZ(), BitString("01010000")), data_qubits[8], c_bits[1:8]...)
        push!(c, IfStatement(GateZ(), BitString("11010000")), [data_qubits[5], data_qubits[7]], c_bits[1:8]...)

        push!(c, IfStatement(GateZ(), BitString("00110000")), [data_qubits[3], data_qubits[7]], c_bits[1:8]...)
        push!(c, IfStatement(GateZ(), BitString("10110000")), [data_qubits[2], data_qubits[7]], c_bits[1:8]...)
        push!(c, IfStatement(GateZ(), BitString("01110000")), [data_qubits[3], data_qubits[8]], c_bits[1:8]...)
        push!(c, IfStatement(GateZ(), BitString("11110000")), [data_qubits[2], data_qubits[8]], c_bits[1:8]...)
    end
end

add_surface_code_d3 (generic function with 1 method)

In [4]:
function logical_indexes(i)
    """
    Return the ranges of data_qubits, ancillas and c_bits depending on which logical qubit we look at.
    >>> logical_indexes(1)
    (1:9, 10:17, 1:9)
    """
    data_per_logic = 9
    ancillas_per_logic = 8
    c_bits_per_logic = 9

    data_qubits = (i-1) * (data_per_logic + ancillas_per_logic) + 1 : (i-1) * (data_per_logic + ancillas_per_logic) + data_per_logic
    ancillas = (i-1) * (data_per_logic + ancillas_per_logic) + data_per_logic + 1 : (i) * (data_per_logic + ancillas_per_logic)
    c_bits = (i-1) * data_per_logic + 1 : (i) * data_per_logic

    return (data_qubits, ancillas, c_bits)
end

# function logical_indexes_t(nb_qubits)
#     """
#     Return the range of the qubits specialized on the t gate.
#     Those qubits are basically the 10 lasts qubits.
#     The last qubit is the auxiliary qubit for quantum teleportation.
#     Also return classical bits for quantum teleportation
#     >>> logical_indexes_t(2)
#     35:44, 19:20
#     """
    
#     nb_qubits += 1
#     i = nb_qubits

#     data_per_logic = 9
#     ancillas_per_logic = 8
#     c_bits_per_logic = 9

#     data = (i-1) * (data_per_logic + ancillas_per_logic) + 1 : (i-1) * (data_per_logic + ancillas_per_logic) + data_per_logic + 1
#     c_bits = (i-1) * data_per_logic + 1 : (i) * data_per_logic

#     return data, c_bits[1:2]
# end

logical_indexes (generic function with 1 method)

In [5]:
logical_indexes(1), logical_indexes(2), logical_indexes(3)#, logical_indexes_t(2)

((1:9, 10:17, 1:9), (18:26, 27:34, 10:18), (35:43, 44:51, 19:27))

In [6]:
function logical_circuit()
    return [Circuit(), Circuit()]
end

function logical_draw(c)
    if typeof(c) == typeof(Circuit())
        drawable = nothing
    else
        drawable = c[2]
        c = c[1]
    end
    if drawable != nothing
        draw(drawable)
    end
end

function logical_execute(conn, c; label="logical", algorithm="auto", nsamples=1000, bitstrings=BitString[], timelimit=30, bonddim=256, entdim=16, seed=rand(Int))
    if typeof(c) == typeof(Circuit())
        drawable = nothing
    else
        drawable = c[2]
        c = c[1]
    end
    return execute(conn, c, label=label, algorithm=algorithm, nsamples=nsamples, bitstring=bitstring, timelimit=timelimit, bonddim=bonddim, entdim=entdim, seed=seed)
end

function logical_x(c, l_target)
    # X on logical qubit with surface code = apply X on all qubits of a column
    data_qubits, _, _ = logical_indexes(l_target)
    if typeof(c) == typeof(Circuit())
        drawable = nothing
    else
        drawable = c[2]
        c = c[1]
    end
    push!(c, GateX(), [data_qubits[2], data_qubits[5], data_qubits[8]])
    if drawable != nothing
        push!(drawable, GateX(), l_target)
    end
end

function logical_z(c, l_target)
    # Z on logical qubit with surface code = apply Z on all qubits of a row
    data_qubits, _, _ = logical_indexes(l_target)
    if typeof(c) == typeof(Circuit())
        drawable = nothing
    else
        drawable = c[2]
        c = c[1]
    end
    push!(c, GateZ(), [data_qubits[4], data_qubits[5], data_qubits[6]])
    if drawable != nothing
        push!(drawable, GateZ(), l_target)
    end
end

function logical_t(c, l_target, nb_qubits)
    """
    Apply a logical T gate.
    For now, the method used is very simple: apply T gate on the middle data qubit.
    It strangly works...

    Tests have been made for H T H (measure ~85% |0>), and H T T T T H (= H Z H) (measure 100% |1>)
    """
    data_qubits, _, _ = logical_indexes(l_target)
    if typeof(c) == typeof(Circuit())
        drawable = nothing
    else
        drawable = c[2]
        c = c[1]
    end

    push!(c, GateT(), data_qubits[5])

    # t_qubits, t_bits = logical_indexes_t(nb_qubits)

    # # Prepare t qubits (H + T)
    # push!(c, GateH(), t_qubits[1:9])
    # push!(c, GateT(), t_qubits[1:9])
    # auxilliary = t_qubits[10]
    
    # # For each data_qubit
    # for (data_qubit, t_qubit) in zip(data_qubits, t_qubits)
    #     # Use teleportation between data_qubit and t_qubit, with auxilliary
    #     # Reset auxilliary
    #     push!(c, Reset(), auxilliary)

    #     # Do the quantum teleportation circuit but with H+T instead of |0>
    #     push!(c, GateH(), auxilliary)
    #     push!(c, GateCX(), auxilliary, t_qubit)
    #     push!(c, GateCX(), data_qubit, auxilliary)
    #     push!(c, GateH(), data_qubit)

    #     # Measure auxilliary and data qubit
    #     push!(c, Measure(), [auxilliary, data_qubit], t_bits)

    #     push!(c, IfStatement(GateX(), BitString("1")), t_qubit, t_bits[1])
    #     push!(c, IfStatement(GateZ(), BitString("1")), t_qubit, t_bits[2])

    #     # Swap to get result in data_qubit
    #     push!(c, GateSWAP(), data_qubit, t_qubit)
    # end

    if drawable != nothing
        push!(drawable, GateT(), l_target)
    end
end

function logical_swap(c, l_target1, l_target2)
    # Swap every data_qubits and ancillas
    data_t1, ancilla_t1, _ = logical_indexes(l_target1)
    data_t2, ancilla_t2, _ = logical_indexes(l_target2)
    if typeof(c) == typeof(Circuit())
        drawable = nothing
    else
        drawable = c[2]
        c = c[1]
    end
    for (physical_t1, physical_t2) in zip(data_t1, data_t2)
        push!(c, GateSWAP(), physical_t1, physical_t2)
    end
    for (physical_t1, physical_t2) in zip(ancilla_t1, ancilla_t2)
        push!(c, GateSWAP(), physical_t1, physical_t2)
    end
    if drawable != nothing
        push!(drawable, GateSWAP(), l_target1, l_target2)
    end
end

function logical_measure(c, l_target)
    # Measure all qubits of the logical qubit, and set results in the associated classical bits
    data_qubits, _, c_bits = logical_indexes(l_target)
    if typeof(c) == typeof(Circuit())
        drawable = nothing
    else
        drawable = c[2]
        c = c[1]
    end
    for (qbit, cbit) in zip(data_qubits, c_bits)
        push!(c, Measure(), qbit, cbit)
    end
    if drawable != nothing
        push!(drawable, Measure(), l_target, l_target)
    end
end

function correct_logical_qubit(c, l_target)
    # Apply surface code for the logical qubit (stabilize state |0>)
    data_qubits, ancillas, c_bits = logical_indexes(l_target)
    if typeof(c) == typeof(Circuit())
        drawable = nothing
    else
        drawable = c[2]
        c = c[1]
    end
    add_surface_code_d3(c, data_qubits, ancillas, c_bits)
end

function logical_cx(c, l_control, l_target)
    # Apply a CX between 2 logical qubits (= apply CX on each pair of data_qubits)
    data_control, _, _ = logical_indexes(l_control)
    data_target, _, _ = logical_indexes(l_target)
    if typeof(c) == typeof(Circuit())
        drawable = nothing
    else
        drawable = c[2]
        c = c[1]
    end
    for (physical_control, physical_target) in zip(data_control, data_target)
        push!(c, GateCX(), physical_control, physical_target)
    end
    if drawable != nothing
        push!(drawable, GateCX(), l_control, l_target)
    end
end

function logical_h(c, l_target)
    # Apply H on all physical qubits, and rotate by 90°
    data_qubits, _, _ = logical_indexes(l_target)
    if typeof(c) == typeof(Circuit())
        drawable = nothing
    else
        drawable = c[2]
        c = c[1]
    end
    for i in data_qubits
        push!(c, GateH(), i)
    end
    """
    987    789    189    389
    654 -> 654 -> 654 -> 654
    321    321    327    127
    """
    push!(c, GateSWAP(), data_qubits[9], data_qubits[7])
    push!(c, GateSWAP(), data_qubits[7], data_qubits[1])
    push!(c, GateSWAP(), data_qubits[1], data_qubits[3])
    """
    389    349    329    369
    654 -> 658 -> 658 -> 258
    127    127    147    147
    """
    push!(c, GateSWAP(), data_qubits[8], data_qubits[4])
    push!(c, GateSWAP(), data_qubits[4], data_qubits[2])
    push!(c, GateSWAP(), data_qubits[2], data_qubits[6])
    if drawable != nothing
        push!(drawable, GateH(), l_target)
    end
end

logical_h (generic function with 1 method)

In [7]:
function count_bitstrings(n, res)
    # Initialize an empty dictionary to store the bitstrings and their counts
    bitstring_counts = Dict{String, Int64}()
    
    # Loop over the samples and count the bitstrings
    for (bs, val) in histsamples(res)
        logical_bitstring = ""
        for i in 1:n
            # Count number of 1 in the data_qubits of the ith qubit
            nb_ones = count(x -> x == '1', string(bs[(i-1)*9 + 1 : i*9]))

            if nb_ones & 1 == 0
                logical_bitstring *= "0" # even number of ones => 0
            else
                logical_bitstring *= "1" # odd number of ones => 1
            end
        end

        if !haskey(bitstring_counts, logical_bitstring)
            bitstring_counts[logical_bitstring] = val
        else
            bitstring_counts[logical_bitstring] += val
        end
    end
    
    return bitstring_counts
end

count_bitstrings (generic function with 1 method)

---
# Testing

## Test GHZ state with error

In [8]:
c = logical_circuit()
correct_logical_qubit(c, 1)
correct_logical_qubit(c, 2)
logical_h(c, 1)
logical_cx(c, 1, 2)
logical_measure(c, 1)
logical_measure(c, 2)

add_noise!(c[1], GateH(), AmplitudeDamping(0.5))
add_noise!(c[2], GateH(), AmplitudeDamping(0.5))

logical_draw(c)

       ┌─┐┌─────────────────────┐   ┌─┐                                         
q[1]: ╶┤H├┤AmplitudeDamping(0.5)├─●─┤M├───╴                                     
       └─┘└─────────────────────┘┌┴┐└╥┘┌─┐                                      
q[2]: ╶──────────────────────────┤X├─╫─┤M├╴                                     
                                 └─┘ ║ └╥┘                                      
                                     ║  ║                                       
c:    ═══════════════════════════════╩══╩═                                      
                                     1  2                                       



In [9]:
job_logical = execute(conn, c[2], algorithm="mps", nsamples=1000, label="logical")
res_logical = getresult(conn, job_logical)

job = logical_execute(conn, c, algorithm="mps", nsamples=100, label="physical_corrected")
res = getresult(conn, job)

n = numqubits(c[2])
println("Physical_corrected: ", count_bitstrings(n, res))

println("Logical results: ", histsamples(res_logical))

Physical_corrected: Dict("00" => 51, "11" => 49)
Logical results: Dict{BitString, Int64}(bs"00" => 734, bs"11" => 266)


---
## Testing T gate

In [10]:
c = logical_circuit()
correct_logical_qubit(c, 1)
logical_h(c, 1)
logical_t(c, 1, 1)
logical_t(c, 1, 1)
logical_t(c, 1, 1)
logical_t(c, 1, 1)
logical_h(c, 1)
logical_measure(c, 1)

logical_draw(c)

       ┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐                                                    
q[1]: ╶┤H├┤T├┤T├┤T├┤T├┤H├┤M├╴                                                   
       └─┘└─┘└─┘└─┘└─┘└─┘└╥┘                                                    
                          ║                                                     
c:    ════════════════════╩═                                                    
                          1                                                     



In [11]:
job_logical = execute(conn, c[2], algorithm="mps", nsamples=1000, label="logical")
job = logical_execute(conn, c, algorithm="mps", nsamples=100, label="physical_corrected")

res_logical = getresult(conn, job_logical)
res = getresult(conn, job)

n = numqubits(c[2])
println("Physical_corrected: ", count_bitstrings(n, res))

println("Logical results: ", histsamples(res_logical))

Physical_corrected: Dict("1" => 100)
Logical results: Dict{BitString, Int64}(bs"1" => 1000)


In [12]:
c = logical_circuit()
correct_logical_qubit(c, 1)
logical_h(c, 1)
logical_t(c, 1, 1)
logical_h(c, 1)
logical_measure(c, 1)

logical_draw(c)

       ┌─┐┌─┐┌─┐┌─┐                                                             
q[1]: ╶┤H├┤T├┤H├┤M├╴                                                            
       └─┘└─┘└─┘└╥┘                                                             
                 ║                                                              
c:    ═══════════╩═                                                             
                 1                                                              



In [13]:
job_logical = execute(conn, c[2], algorithm="mps", nsamples=1000, label="logical")
job = logical_execute(conn, c, algorithm="mps", nsamples=100, label="physical_corrected")

res_logical = getresult(conn, job_logical)
res = getresult(conn, job)

n = numqubits(c[2])
println("Physical_corrected: ", count_bitstrings(n, res))

println("Logical results: ", histsamples(res_logical))

Physical_corrected: Dict("1" => 15, "0" => 85)
Logical results: Dict{BitString, Int64}(bs"1" => 141, bs"0" => 859)
